# TrainWatcher: Getting Started

This notebook shows the simplest way to send training notifications with TrainWatcher.
The PyPI distribution name is `trainwatcher`, and the import package is `trainwatcher`.

In [ ]:
# If TrainWatcher is not installed yet:
# !pip install trainwatcher

## Cloud email setup (recommended)
Most users can just register their email once.
If you self-host the Worker, set TRAINWATCHER_BASE_URL first.

In [ ]:
from trainwatcher import add_email

# Run once to register your email and save the API key locally.
# add_email("you@example.com")

# Self-host only:
# import os
# os.environ["TRAINWATCHER_BASE_URL"] = "https://your-worker.workers.dev"

## Basic training run (manual try/except)

In [ ]:
from trainwatcher import monitor

def train_model(epochs=3):
    for epoch in range(1, epochs + 1):
        loss = 0.6 / epoch
        val_acc = 0.75 + (epoch * 0.03)
        monitor.log(epoch=epoch, loss=loss, val_acc=val_acc)

monitor.start()
try:
    train_model(epochs=5)
    monitor.end()
except Exception as exc:
    monitor.fail(exc)
    raise

## Context manager (auto end/fail)

In [ ]:
from trainwatcher import monitor

def train_quick():
    for epoch in range(1, 4):
        monitor.log(epoch=epoch, loss=0.5 / epoch)

with monitor.watch():
    train_quick()

## Mid-run notifications

In [ ]:
import time
from trainwatcher import monitor

monitor.start()
monitor.heartbeat(interval_seconds=60, message="Training still running")

for epoch in range(1, 4):
    time.sleep(1)
    monitor.step(notify_every=2, message=f"Epoch {epoch} completed")

monitor.end()

## Failure example (sends "Training Failed")

In [ ]:
from trainwatcher import monitor

monitor.start()
try:
    # Uncomment to test failure notifications:
    # raise RuntimeError("Intentional failure example")
    monitor.end()
except Exception as exc:
    monitor.fail(exc)
    raise